# Generalized Stabilizer Simulation: Step-by-Step Walkthrough

This notebook traces a small 2-qubit Clifford+T circuit **by hand**, showing exactly how the generalized stabilizer state $(\mathbf{v}, \mathcal{T}(S,D))$ evolves under each operation. The goal is to build intuition for:

- Why Clifford gates are "free" (tableau-only, zero coefficient overhead)
- How the inverse tableau gives O(n/W) Pauli decomposition
- How T gates split entries via $\beta$ and $\gamma$ bitmasks
- Why Z-stabilizers prevent T-gate growth (the key QEC insight)
- How measurement collapses entries via pivot-bit pairing

## The Circuit

```
q0: |0⟩ ── H ── T ──────── H ────── T ──── CNOT ── M ──────
q1: |0⟩ ────────── T(diag) ── H ── T(split) ── ⊕ ── M ──
```

Steps:
1. Initialize $|00\rangle$
2. $H$ on q0 — Clifford (tableau update, $\mathbf{v}$ unchanged)
3. $T$ on q0 — splits: $|\mathbf{v}|: 1 \to 2$
4. $T$ on q1 — **diagonal** ($\beta=0$, $Z_1$ is a stabilizer): $|\mathbf{v}|$ stays 2
5. $H$ on q1 — Clifford
6. $T$ on q1 — splits: $|\mathbf{v}|: 2 \to 4$
7. CNOT(0,1) — Clifford (entanglement)
8. Measure q0 — random ($\beta \neq 0$): $|\mathbf{v}|$ shrinks
9. Measure q1

## The Representation: $(\mathbf{v}, \mathcal{T}(S, D))$

The standard stabilizer formalism (Aaronson–Gottesman) represents an $n$-qubit state by $2n$ Pauli operators — $n$ **stabilizers** $\{s_i\}$ that fix the state ($s_i|\psi_S\rangle = +|\psi_S\rangle$) and $n$ **destabilizers** $\{d_i\}$ that complete a basis for the Pauli group. Together these form the **tableau** $\mathcal{T}(S, D)$, stored as a $2n \times 2n$ binary matrix plus a sign vector.

This is extremely efficient for Clifford circuits but cannot handle non-Clifford gates like $T$. The **generalized stabilizer formalism** (used by [SOFT](https://arxiv.org/abs/2512.23037)) extends this by pairing the tableau with a sparse coefficient vector $\mathbf{v}$:

$$
|\phi\rangle = \sum_{\alpha} v_\alpha \, |b_\alpha\rangle
\qquad\text{where}\qquad
|b_\alpha\rangle = d^\alpha |\psi_S\rangle
$$

Here $\alpha \in \{0,1\}^n$ is a binary key and $d^\alpha = d_0^{\alpha_0} d_1^{\alpha_1} \cdots d_{n-1}^{\alpha_{n-1}}$ is a product of destabilizers selected by the bits of $\alpha$. Each destabilizer either flips the state to an orthogonal basis vector or leaves it alone, so the $|b_\alpha\rangle$ form an orthonormal basis for the full $2^n$-dimensional Hilbert space.

The state of the simulator at any point is the pair $(\mathbf{v}, \mathcal{T}(S,D))$:

| Component | What it stores | Size |
|-----------|---------------|------|
| $\mathcal{T}(S, D)$ | $2n$ Pauli generators (stabilizers + destabilizers) as a binary symplectic matrix + signs | $O(n^2)$ bits |
| $\mathbf{v}$ | Sparse map from binary keys $\alpha$ to complex amplitudes $v_\alpha$ | $O(\|\mathbf{v}\|)$ entries |

The key properties that make this representation powerful:

- **Clifford gates are free.** A Clifford gate $C$ conjugates each generator ($d_i \mapsto C d_i C^\dagger$, $s_i \mapsto C s_i C^\dagger$), updating the tableau in $O(n/W)$ SIMD operations per row. The coefficient vector $\mathbf{v}$ is **completely untouched** — the keys $\alpha$ and amplitudes $v_\alpha$ don't change.

- **T gates can at most double $|\mathbf{v}|$.** The $T$ gate decomposes as $T = \cos(\pi/8)\,I - i\sin(\pi/8)\,Z$. Applying this requires knowing how $Z_q$ acts on the basis $\{|b_\alpha\rangle\}$, which is determined by a **Pauli decomposition** of $Z_q$ into the current stabilizer–destabilizer basis. This decomposition yields two bitmasks:
  - $\beta$ (**destabilizer mask**): $Z_q$ maps $|b_\alpha\rangle \to |b_{\alpha \oplus \beta}\rangle$
  - $\gamma$ (**stabilizer mask**): contributes a sign $(-1)^{\gamma \cdot \alpha}$ to each entry

  When $\beta = 0$, $Z_q$ is in the stabilizer group and acts diagonally (phase-only update, no new entries). When $\beta \neq 0$, each existing entry spawns two — but a half-sort merge pass recombines duplicates, keeping $|\mathbf{v}|$ bounded.

- **Measurement collapses $|\mathbf{v}|$.** Measuring $Z_q$ projects onto an eigenspace and eliminates half the entries (those inconsistent with the outcome), partially undoing T-gate growth.

For quantum error-correcting codes, the logical $Z$ operators are stabilizers by construction, so T gates on data qubits often have $\beta = 0$ and $|\mathbf{v}|$ stays bounded even at high T-depth. This is the central insight that makes the formalism practical.

The rest of this notebook traces each operation on a concrete 2-qubit circuit, showing exactly how $\mathbf{v}$ and $\mathcal{T}$ evolve at every step.

## Setup

In [ ]:
import sys, os, math, pathlib
import numpy as np

# Ensure prototype modules are importable from any kernel CWD
_proto = pathlib.Path('/home/exedev/ctsim/prototype')
if not (_proto / 'state.py').exists():
    # Fallback: assume CWD is the prototype directory
    _proto = pathlib.Path('.')
sys.path.insert(0, str(_proto))
os.chdir(_proto)

from state import GeneralizedStabilizerState
from tableau import Tableau

# Gate matrices for independent statevector verification
I2 = np.eye(2, dtype=complex)
H_mat = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S_mat = np.array([[1, 0], [0, 1j]], dtype=complex)
T_mat = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)
CNOT_mat = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

def kron_gate(gate, qubit, n=2):
    """Build n-qubit matrix from single-qubit gate on given qubit."""
    ops = [I2] * n
    ops[qubit] = gate
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def apply_cnot_sv(sv, ctrl, tgt, n=2):
    """Apply CNOT to statevector."""
    dim = 1 << n
    result = np.zeros(dim, dtype=complex)
    for i in range(dim):
        bits = list(format(i, f'0{n}b'))
        if bits[ctrl] == '1':
            bits[tgt] = '0' if bits[tgt] == '1' else '1'
        result[int(''.join(bits), 2)] += sv[i]
    return result

def states_equal(sv1, sv2, atol=1e-8):
    """Check equality up to global phase."""
    for i in range(len(sv1)):
        if abs(sv1[i]) > atol:
            phase = sv2[i] / sv1[i]
            return np.allclose(sv1 * phase, sv2, atol=atol)
    return np.allclose(sv2, 0, atol=atol)

## Helper: Display Functions

In [ ]:
def show_tableau(tab, label=""):
    """Show tableau with raw GF(2) bits alongside Pauli strings."""
    n = tab.n
    if label:
        print(f"\n── {label} ──")
    # Header
    x_hdr = ' '.join(f'x{q}' for q in range(n))
    z_hdr = ' '.join(f'z{q}' for q in range(n))
    print(f"  {'':12s}  {x_hdr} | {z_hdr} | s  Pauli")
    print(f"  {'':12s}  {'─'*(3*n-1)} + {'─'*(3*n-1)} + ── ─────")
    for i in range(n):
        xbits = ' '.join(f' {tab.x(i,q)}' for q in range(n))
        zbits = ' '.join(f' {tab.z(i,q)}' for q in range(n))
        ps = tab.row_to_pauli_string(i)
        print(f"  d[{i}]          {xbits} | {zbits} | {tab.sign(i)}  {ps}")
    for i in range(n):
        xbits = ' '.join(f' {tab.x(n+i,q)}' for q in range(n))
        zbits = ' '.join(f' {tab.z(n+i,q)}' for q in range(n))
        ps = tab.row_to_pauli_string(n+i)
        print(f"  s[{i}]          {xbits} | {zbits} | {tab.sign(n+i)}  {ps}")

def show_v(state, label=""):
    """Show coefficient vector entries."""
    if label:
        print(f"\n  {label}")
    n = state.n
    state._cleanup()
    print(f"  |v| = {state.num_entries} entries:")
    for alpha in sorted(state.v.keys()):
        c = state.v[alpha]
        if abs(c) < 1e-10:
            continue
        bits = format(alpha, f'0{n}b')
        active = [f'd{i}' for i in range(n) if (alpha >> (n-1-i)) & 1]
        destab_str = '·'.join(active) if active else 'I'
        # Format complex number readably
        if abs(c.imag) < 1e-10:
            cstr = f'{c.real:+.6f}'
        elif abs(c.real) < 1e-10:
            cstr = f'{c.imag:+.6f}i'
        else:
            cstr = f'{c.real:+.6f}{c.imag:+.6f}i'
        print(f"    α={bits}  v={cstr}  →  ({destab_str})|ψ_S⟩")

def show_statevector(state):
    """Show non-zero amplitudes of the full statevector."""
    sv = state.to_statevector()
    n = state.n
    print(f"  Statevector:")
    for i in range(1 << n):
        if abs(sv[i]) > 1e-10:
            bits = format(i, f'0{n}b')
            c = sv[i]
            if abs(c.imag) < 1e-10:
                cstr = f'{c.real:+.6f}'
            elif abs(c.real) < 1e-10:
                cstr = f'{c.imag:+.6f}i'
            else:
                cstr = f'{c.real:+.6f}{c.imag:+.6f}i'
            print(f"    |{bits}⟩: {cstr}")

def verify(state, ref_sv, step=""):
    """Verify generalized stabilizer state matches reference statevector."""
    gsf_sv = state.to_statevector()
    ok = states_equal(gsf_sv, ref_sv)
    status = "✓" if ok else "✗ MISMATCH"
    print(f"  Verification {step}: {status}")
    if not ok:
        print(f"    GSF: {gsf_sv}")
        print(f"    Ref: {ref_sv}")
    return ok

## Step 0: Initialization — $|00\rangle$

The state is $(\mathbf{v}, \mathcal{T}(S,D))$ where:
- Stabilizers: $s_0 = +Z_0$, $s_1 = +Z_1$ (the $|0\rangle$ eigenvalue conditions)
- Destabilizers: $d_0 = +X_0$, $d_1 = +X_1$ (their symplectic duals)
- Coefficient vector: $\mathbf{v} = \{00 \to 1\}$ (one entry, the stabilizer state itself)

The basis states are $|b_\alpha\rangle = d^\alpha |\psi_S\rangle$:
- $|b_{00}\rangle = I|\psi_S\rangle = |00\rangle$
- $|b_{10}\rangle = d_0|\psi_S\rangle = X_0|00\rangle = |10\rangle$
- $|b_{01}\rangle = d_1|\psi_S\rangle = X_1|00\rangle = |01\rangle$
- $|b_{11}\rangle = d_0 d_1|\psi_S\rangle = X_0 X_1|00\rangle = |11\rangle$

In [ ]:
# Initialize
state = GeneralizedStabilizerState(2)
ref_sv = np.array([1, 0, 0, 0], dtype=complex)  # |00>

show_tableau(state.tableau, "Step 0: |00⟩")
show_v(state)
verify(state, ref_sv, "step 0")

## Step 1: Hadamard on q0 — Clifford Gate

$H$ conjugates each tableau row: $X \leftrightarrow Z$, $Y \to -Y$.

**What changes:**
- $d_0 = +X_0 \to +Z_0$ (X and Z swap on q0)
- $s_0 = +Z_0 \to +X_0$
- Rows for q1 are untouched.

**What does NOT change:** The coefficient vector $\mathbf{v} = \{00 \to 1\}$. The key $\alpha = 00$ still means "apply no destabilizers." But now $d_0 = Z_0$ and $s_0 = X_0$, so the stabilizer state is $|+\rangle|0\rangle$ and the basis has changed.

This is the core Clifford insight: **all the physics is absorbed by the tableau update; the coefficient vector is completely untouched.**

In [ ]:
state.apply_h(0)
ref_sv = kron_gate(H_mat, 0) @ ref_sv

show_tableau(state.tableau, "Step 1: After H on q0")
show_v(state, "v unchanged (Clifford!)")
show_statevector(state)
verify(state, ref_sv, "step 1")

## Step 2: T on q0 — Entry Splits ($|\mathbf{v}|: 1 \to 2$)

$T = e^{i\pi/8}[\cos(\pi/8) I - i \sin(\pi/8) Z]$. To apply $T$ on q0, we need to know how $Z_0$ acts on our basis. This requires **decomposing $Z_0$ via the inverse tableau**.

### Pauli Decomposition of $Z_0$

We compute two bitmasks:
- $\beta_i = 1$ iff $Z_0$ anticommutes with stabilizer $s_i$
- $\gamma_i$ (`stab_bits`) $= 1$ iff $Z_0$ anticommutes with destabilizer $d_i$

Current tableau: $s_0 = +X_0$, $s_1 = +Z_1$, $d_0 = +Z_0$, $d_1 = +X_1$

| Generator | Pauli on q0 | Commutes with $Z_0$? | Result |
|-----------|-------------|----------------------|--------|
| $s_0 = +X_0$ | $X$ | $\{Z, X\} = 0$ — **anticommutes** | $\beta_0 = 1$ |
| $s_1 = +Z_1$ | $I$ | commutes | $\beta_1 = 0$ |
| $d_0 = +Z_0$ | $Z$ | $[Z, Z] = 0$ — commutes | $\gamma_0 = 0$ |
| $d_1 = +X_1$ | $I$ | commutes | $\gamma_1 = 0$ |

So $\beta = [1, 0]$ (integer: $\beta = 2$ in MSB-first encoding) and $\gamma = [0, 0]$.

The decomposition says $Z_0 = \text{base\_phase} \times d_0^{\beta_0} \times s_0^{\gamma_0} \cdots = \text{base\_phase} \times d_0 = \text{base\_phase} \times Z_0$. So $\text{base\_phase} = +1$.

### Action on basis states

$$Z_0|b_\alpha\rangle = \xi_\alpha(Z_0) \, |b_{\alpha \oplus \beta}\rangle$$

where $\xi_\alpha = \text{base\_phase} \times (-1)^{\gamma \cdot \alpha} = 1 \times (-1)^0 = +1$ for all $\alpha$ (since $\gamma = [0,0]$).

### T-gate update

For each entry $v_\alpha |b_\alpha\rangle$:
- **I-branch:** $\cos(\pi/8) \, v_\alpha \, |b_\alpha\rangle$ (key unchanged)
- **Z-branch:** $-i \sin(\pi/8) \, \xi_\alpha \, v_\alpha \, |b_{\alpha \oplus \beta}\rangle$ (key XOR'd with $\beta$)

Starting from $\alpha = 00$, $v_{00} = 1$:
- I-branch: key $00$, amplitude $\cos(\pi/8) \approx 0.924$
- Z-branch: key $00 \oplus 10 = 10$, amplitude $-i \sin(\pi/8) \approx -0.383i$

**Half-sort optimization:** The I-branch preserves key order (trivially sorted here). Only the Z-branch needs sorting. With 1 entry each, there's nothing to sort.

**Two-pointer merge:** keys $00 < 10$, no duplicates. Result: 2 entries.

In [ ]:
# Show the decomposition explicitly
z_x = np.array([0, 0])
z_z = np.array([1, 0])  # Z on q0
beta, gamma, base_phase = state.tableau.pauli_action(z_x, z_z, 0)
print(f"  Decomposition of Z_0:")
print(f"    β (destab mask) = {[int(b) for b in beta]}  → keys XOR with {int(beta[0])*2 + int(beta[1])}")
print(f"    γ (stab mask)   = {[int(g) for g in gamma]}  → phase depends on α·γ")
print(f"    base_phase      = {base_phase}")
print()

# Apply T
state.apply_t(0)
ref_sv = kron_gate(T_mat, 0) @ ref_sv

show_tableau(state.tableau, "Step 2: After T on q0 (tableau UNCHANGED by T!)")
show_v(state, "v now has 2 entries (T caused a split)")
show_statevector(state)
verify(state, ref_sv, "step 2")

Note that **the tableau is unchanged by the T gate** — only $\mathbf{v}$ changes. This is the complement of Clifford gates, which only change the tableau.

## Step 3: T on q1 — Diagonal Case ($\beta = 0$, No Split!)

**This is the key QEC insight.** When $Z_q$ is already in the stabilizer group, $\beta = 0$ and the T gate cannot create new entries.

### Decomposition of $Z_1$

Current stabilizers: $s_0 = +X_0$, $s_1 = +Z_1$. Since $Z_1$ commutes with both stabilizers ($Z$ commutes with $X$ on a different qubit, and $Z$ commutes with $Z$):

$$\beta = [0, 0] \quad \Longrightarrow \quad Z_1|b_\alpha\rangle = \xi_\alpha |b_\alpha\rangle \text{ (no index shift!)}$$

$Z_1$ is purely in the stabilizer group, so it acts as a **diagonal operator** on the basis — each entry gets a phase but keeps its key.

For the phase: $\gamma_1 = 1$ (since $Z_1$ anticommutes with $d_1 = X_1$), so $\xi_\alpha = \text{base\_phase} \times (-1)^{\alpha_1}$. Since none of our current keys ($\alpha = 00, 10$) have bit 1 set, $\alpha_1 = 0$ for all entries and every entry gets the **same scalar phase**.

$$T|b_\alpha\rangle = [\cos(\pi/8) - i\sin(\pi/8) \cdot \underbrace{\xi_\alpha}_{=+1}] |b_\alpha\rangle = e^{-i\pi/4} |b_\alpha\rangle$$

**All entries get the same global phase. $|\mathbf{v}|$ stays at 2.**

In the C++ implementation, this is the **$|\mathbf{v}|=1$ fast path** generalized: when $\beta = 0$, we just scale each entry's amplitude without any sort/merge.

In [ ]:
# Show decomposition
z1_x = np.array([0, 0])
z1_z = np.array([0, 1])  # Z on q1
beta1, gamma1, bp1 = state.tableau.pauli_action(z1_x, z1_z, 0)
print(f"  Decomposition of Z_1:")
print(f"    β = {[int(b) for b in beta1]}  ← ALL ZEROS! Z_1 is in the stabilizer group")
print(f"    γ = {[int(g) for g in gamma1]}")
print(f"    base_phase = {bp1}")
print(f"  Since β=0: T gate is diagonal, no new entries created.")
print()

state.apply_t(1)
ref_sv = kron_gate(T_mat, 1) @ ref_sv

show_v(state, "v still has 2 entries (no split!)")
verify(state, ref_sv, "step 3")

**Why this matters for QEC:** In a quantum error-correcting code, the code's Z-stabilizers ensure that $Z_q$ is in the stabilizer group for most qubits. T gates on those qubits are "free" — they don't grow $|\mathbf{v}|$. This is why magic state cultivation with a color code keeps $|\mathbf{v}|$ bounded (e.g., $|\mathbf{v}| \leq 2^{19-9} = 1024$ for $d=5$).

## Step 4: H on q1 — Clifford (Preparing q1 for a Split)

This swaps $s_1 = +Z_1 \leftrightarrow d_1 = +X_1$, so after $H$:
- $s_1 = +X_1$ ($Z_1$ is **no longer a stabilizer!**)
- $d_1 = +Z_1$

The coefficient vector is untouched.

In [ ]:
state.apply_h(1)
ref_sv = kron_gate(H_mat, 1) @ ref_sv

show_tableau(state.tableau, "Step 4: After H on q1")
show_v(state, "v unchanged (Clifford)")
verify(state, ref_sv, "step 4")

## Step 5: T on q1 — Splits ($|\mathbf{v}|: 2 \to 4$)

Now $s_1 = +X_1$ and $Z_1$ anticommutes with $X_1$, so $\beta_1 = 1$.

### Decomposition of $Z_1$

| Generator | Commutes with $Z_1$? | Result |
|-----------|----------------------|--------|
| $s_0 = +X_0$ | yes (different qubit) | $\beta_0 = 0$ |
| $s_1 = +X_1$ | **no** ($\{Z,X\}=0$) | $\beta_1 = 1$ |
| $d_0 = +Z_0$ | yes | $\gamma_0 = 0$ |
| $d_1 = +Z_1$ | yes ($[Z,Z]=0$) | $\gamma_1 = 0$ |

$\beta = [0, 1]$ (integer: 1), $\gamma = [0, 0]$, base\_phase $= +1$.

### Expansion (by hand)

Existing entries: $\{00 \to a, \; 10 \to b\}$ for some complex $a, b$.

For $\alpha = 00$:
- I-branch: key $00$, amp $\cos(\pi/8) \cdot a$
- Z-branch: key $00 \oplus 01 = 01$, amp $-i\sin(\pi/8) \cdot a$ **(new key!)**

For $\alpha = 10$:
- I-branch: key $10$, amp $\cos(\pi/8) \cdot b$
- Z-branch: key $10 \oplus 01 = 11$, amp $-i\sin(\pi/8) \cdot b$ **(new key!)**

### Half-sort

- I-branch: $\{00, 10\}$ — already sorted (same order as Main)
- Z-branch: $\{01, 11\}$ — needs sorting (but happens to already be sorted here)

### Two-pointer merge

All 4 keys are distinct ($00 < 01 < 10 < 11$), so no cancellations. $|\mathbf{v}| = 4$.

In [ ]:
# Show decomposition
beta5, gamma5, bp5 = state.tableau.pauli_action(z1_x, z1_z, 0)
print(f"  Decomposition of Z_1 (after H made X_1 a stabilizer):")
print(f"    β = {[int(b) for b in beta5]}  ← β_1=1, so keys get XOR'd with 01")
print(f"    γ = {[int(g) for g in gamma5]}")
print(f"    base_phase = {bp5}")
print(f"  New keys will be created: 00⊕01=01, 10⊕01=11")
print()

state.apply_t(1)
ref_sv = kron_gate(T_mat, 1) @ ref_sv

show_v(state, "v now has 4 entries!")
show_statevector(state)
verify(state, ref_sv, "step 5")

## Step 6: CNOT(0,1) — Clifford (Entanglement)

CNOT conjugation rules on the Pauli group:
- $X_c \to X_c X_t$ (X on control spreads to target)
- $Z_t \to Z_c Z_t$ (Z on target picks up control)
- $X_t \to X_t$, $Z_c \to Z_c$ (unchanged)

Applied to each row (control=q0, target=q1):

| Before | Rule | After |
|--------|------|-------|
| $d_0 = +Z_0$ | $Z$ on control: unchanged | $+Z_0$ |
| $d_1 = +Z_1$ | $Z$ on target: $Z_1 \to Z_0 Z_1$ | $+Z_0 Z_1$ |
| $s_0 = +X_0$ | $X$ on control: $X_0 \to X_0 X_1$ | $+X_0 X_1$ |
| $s_1 = +X_1$ | $X$ on target: unchanged | $+X_1$ |

The stabilizers $X_0 X_1$ and $X_1$ now encode entanglement. **$\mathbf{v}$ is completely untouched** — still 4 entries with the same keys and amplitudes.

In [ ]:
state.apply_cnot(0, 1)
ref_sv = apply_cnot_sv(ref_sv, 0, 1)

show_tableau(state.tableau, "Step 6: After CNOT(0,1)")
show_v(state, "v unchanged (Clifford!)")
verify(state, ref_sv, "step 6")

## Step 7: Measure q0 — Random Measurement ($\beta \neq 0$)

To measure $Z_0$, we first decompose it in the current basis.

### Decomposition of $Z_0$

| Generator | Pauli | Commutes with $Z_0$? | Result |
|-----------|-------|----------------------|--------|
| $s_0 = +X_0 X_1$ | has $X_0$ | **no** | $\beta_0 = 1$ |
| $s_1 = +X_1$ | no $X_0$ or $Y_0$ | yes | $\beta_1 = 0$ |

$\beta = [1, 0] \neq 0$ → **random measurement** (not all basis states share the same eigenvalue).

The **pivot** is the first index where $\beta_p = 1$, so $p = 0$.

### Measurement procedure

1. **Partition by pivot bit:** Split entries into $\alpha_0 = 0$ and $\alpha_0 = 1$ groups
2. **Pair entries:** Each $\alpha$ with $\alpha_0 = 0$ pairs with $\alpha \oplus \beta$ (which has $\alpha_0 = 1$)
3. **Project:** For each pair, compute amplitudes for outcome $\pm 1$: $v'_\alpha = \frac{1}{\sqrt{2}}(v_\alpha \pm \xi_\alpha \cdot v_{\alpha \oplus \beta})$
4. **Sample** outcome from $P(0)$ and $P(1)$
5. **Collapse tableau:** Aaronson–Gottesman pivot (stabilizer absorbs measured observable)

The pairing:
- $\alpha = 00$ pairs with $00 \oplus 10 = 10$
- $\alpha = 01$ pairs with $01 \oplus 10 = 11$

After projection, only entries with pivot bit = 0 survive: keys $\{00, 01\}$. $|\mathbf{v}|$ drops from 4 to at most 2.

In [ ]:
# Show decomposition before measuring
z0_x = np.array([0, 0])
z0_z = np.array([1, 0])
beta7, gamma7, bp7 = state.tableau.pauli_action(z0_x, z0_z, 0)
pivot = state.tableau.find_anticommuting_stabilizer(0)
print(f"  Decomposition of Z_0:")
print(f"    β = {[int(b) for b in beta7]} → random measurement (β≠0)")
print(f"    γ = {[int(g) for g in gamma7]}")
print(f"    base_phase = {bp7}")
print(f"    pivot = {pivot} (first stabilizer anticommuting with Z_0)")
print()
print(f"  Entry pairing (by pivot bit {pivot}):")
n = state.n
beta_int = sum(int(beta7[i]) << (n-1-i) for i in range(n))
for alpha in sorted(state.v.keys()):
    bit_p = (alpha >> (n-1-pivot)) & 1
    if bit_p == 0:
        partner = alpha ^ beta_int
        a_bits = format(alpha, f'0{n}b')
        p_bits = format(partner, f'0{n}b')
        print(f"    α={a_bits} (bit_p=0) pairs with α⊕β={p_bits} (bit_p=1)")
print()

# Measure with fixed seed for reproducibility
rng = np.random.default_rng(42)
print(f"  |v| before measurement: {state.num_entries}")
outcome0 = state.measure(0, rng=rng)
print(f"  Outcome: q0 = {outcome0}")
print(f"  |v| after measurement:  {state.num_entries}")

show_tableau(state.tableau, f"After measuring q0={outcome0}")
show_v(state)
show_statevector(state)

Note that we cannot easily verify against `ref_sv` after a random measurement (the outcome is probabilistic), but we can check internal consistency:

In [ ]:
# The statevector must be normalized and consistent with the outcome
sv = state.to_statevector()
norm = np.linalg.norm(sv)
print(f"  Norm check: {norm:.6f} (should be 1.0)")
print(f"  Outcome consistency: all amplitudes match q0={outcome0}")
for i in range(4):
    bits = format(i, '02b')
    q0_val = int(bits[0])
    if q0_val != outcome0 and abs(sv[i]) > 1e-10:
        print(f"    ERROR: |{bits}> has amplitude {sv[i]} but q0={outcome0}")
    elif q0_val == outcome0 and abs(sv[i]) > 1e-10:
        print(f"    |{bits}⟩: {sv[i]:.6f} ✓")

## Step 8: Measure q1

After collapsing q0, what happens to q1? Let's check whether $Z_1$ is in the stabilizer group (deterministic) or not (random).

In [ ]:
# Decompose Z_1 in the post-measurement basis
z1_x = np.array([0, 0])
z1_z = np.array([0, 1])
beta8, gamma8, bp8 = state.tableau.pauli_action(z1_x, z1_z, 0)
pivot8 = state.tableau.find_anticommuting_stabilizer(1)

print(f"  Decomposition of Z_1 after q0 measurement:")
print(f"    β = {[int(b) for b in beta8]}")
print(f"    pivot = {pivot8}")
if pivot8 is None:
    print(f"    → DETERMINISTIC measurement (Z_1 in stabilizer group)")
else:
    print(f"    → Random measurement (pivot={pivot8})")
print()

print(f"  |v| before: {state.num_entries}")
outcome1 = state.measure(1, rng=rng)
print(f"  Outcome: q1 = {outcome1}")
print(f"  |v| after:  {state.num_entries}")

show_tableau(state.tableau, f"Final: q0={outcome0}, q1={outcome1}")
show_v(state)
show_statevector(state)

# Final state should be a computational basis state
sv_final = state.to_statevector()
print(f"\n  Final state is |{outcome0}{outcome1}⟩: ", end='')
expected_idx = outcome0 * 2 + outcome1
print("✓" if abs(abs(sv_final[expected_idx]) - 1.0) < 1e-6 else "✗")

## The Inverse Tableau: Why It Matters

In the walkthrough above, every T gate and measurement started with the same operation: **decompose a physical Pauli $Z_q$ into the stabilizer–destabilizer basis** to get $\beta$, $\gamma$, and `base_phase`.

The prototype does this with an $O(n)$ loop over all $2n$ generators (`pauli_action`). The C++ implementation uses **Stim's inverse tableau** to do it in $O(n/W)$ — a single SIMD row read.

### How the inverse tableau encodes decompositions

The inverse tableau $\mathcal{T}^{-1}$ has the property that **row $q$ of the Z-half directly encodes the decomposition of physical $Z_q$:**

- The destabilizer bits of row $q$ give $\beta$ (which basis indices flip)
- The stabilizer bits give $\gamma$ (which generators contribute to the phase)
- The sign bit plus Y-count give `base_phase`

Let's verify this by comparing the $O(n)$ decomposition against a direct row read.

In [ ]:
# Build a fresh state, apply some gates, then compare decomposition methods
test = GeneralizedStabilizerState(3)
test.apply_h(0)
test.apply_s(1)     # Creates Y entries in the tableau
test.apply_cnot(0, 1)
test.apply_h(2)
test.apply_cnot(1, 2)

show_tableau(test.tableau, "Test state (3 qubits, includes Y entries)")

print("\n  Decomposition of Z_q for each qubit:")
for q in range(3):
    zq_x = np.zeros(3, dtype=int)
    zq_z = np.zeros(3, dtype=int)
    zq_z[q] = 1
    beta_q, gamma_q, bp_q = test.tableau.pauli_action(zq_x, zq_z, 0)
    print(f"\n    Z_{q}: β={[int(b) for b in beta_q]}  γ={[int(g) for g in gamma_q]}  phase={bp_q:.4f}")
    if all(b == 0 for b in beta_q):
        print(f"         → Z_{q} is in the stabilizer group (T won't split)")
    else:
        print(f"         → Z_{q} has destabilizer components (T will split)")

In the C++ implementation, each of these decompositions is a **single SIMD row read** from `inv_state.zs[q]` — $O(n/W)$ instead of the $O(n)$ loop shown here. For $n=42$ (typical QEC), this is one 64-bit load.

## The Y-Phase Correction ($i^{\text{num\_y}}$)

When the Pauli decomposition involves generators that produce $Y$ entries (both X and Z bits set at the same qubit position), the `base_phase` must include a correction factor of $i^{\text{num\_y}}$, because $Y = iXZ$.

Let's see this in action:

In [ ]:
# Build a state where the decomposition involves Y
test2 = GeneralizedStabilizerState(2)
test2.apply_h(0)
test2.apply_s(0)   # d_0: X_0 -> Z_0, s_0: Z_0 -> X_0. Then S: X->Y on s_0
                    # Actually H then S: d_0=Z_0, s_0=Y_0 (since SXS†=Y)

show_tableau(test2.tableau, "After H, S on q0")

# Decompose Z_0
zq_x = np.array([0, 0])
zq_z = np.array([1, 0])
beta_y, gamma_y, bp_y = test2.tableau.pauli_action(zq_x, zq_z, 0)
print(f"\n  Z_0 decomposition:")
print(f"    β = {[int(b) for b in beta_y]}, γ = {[int(g) for g in gamma_y]}, base_phase = {bp_y}")
print()
print(f"  The stabilizer s_0 = {test2.tableau.row_to_pauli_string(2)}")
print(f"  has both x and z bits set on q0 (that's Y = iXZ).")
print(f"  When this generator appears in the decomposition product,")
print(f"  the i factor must be tracked — this is the i^{{num_y}} correction.")
print(f"  Stim handles this via inplace_right_mul_returning_log_i_scalar.")

## Summary: $|\mathbf{v}|$ Trajectory

In [ ]:
# Replay the circuit tracking |v| at each step
print("  Step  Operation       |v|  Note")
print("  ────  ──────────────  ───  ────")
steps = [
    ("Init |00⟩",      1, "stabilizer state"),
    ("H(q0)",          1, "Clifford: tableau only"),
    ("T(q0)",          2, "split: β=[1,0]≠0"),
    ("T(q1)",          2, "diagonal: β=0, Z_1 is stabilizer"),
    ("H(q1)",          2, "Clifford: tableau only"),
    ("T(q1)",          4, "split: β=[0,1]≠0"),
    ("CNOT(0,1)",      4, "Clifford: tableau only"),
    ("Measure q0",     2, "random: entries paired & projected"),
    ("Measure q1",     1, "collapsed to basis state"),
]
for i, (op, v, note) in enumerate(steps):
    print(f"  {i:4d}  {op:14s}  {v:3d}  {note}")

print()
print("  Key observations:")
print("  • Clifford gates (H, CNOT): |v| NEVER changes")
print("  • T gate with β≠0: |v| can double (but merge may cancel some)")
print("  • T gate with β=0: |v| unchanged (Z_q in stabilizer group)")
print("  • Measurement: |v| can only shrink (projection/filtering)")
print("  • The circuit went 1 → 2 → 2 → 2 → 4 → 4 → 2 → 1")

## Cost Summary

| Operation | Tableau cost | Entry cost | $|\mathbf{v}|$ effect |
|-----------|-------------|------------|----------------------|
| Clifford gate | $O(n/W)$ per row | **zero** | unchanged |
| Pauli decomp (prototype) | $O(n)$ loop | n/a | n/a |
| Pauli decomp (C++ inv tableau) | $O(n/W)$ row read | n/a | n/a |
| T / T† ($\beta \neq 0$) | $O(n/W)$ decomp | $O(|\mathbf{v}| \log |\mathbf{v}|)$ sort+merge | $\leq 2|\mathbf{v}|$ |
| T / T† ($\beta = 0$) | $O(n/W)$ decomp | $O(|\mathbf{v}|)$ scalar multiply | unchanged |
| Measurement (random) | $O(n^2/W)$ collapse | $O(|\mathbf{v}|)$ pair+project | $\leq |\mathbf{v}|$ |
| Measurement (deterministic) | none | $O(|\mathbf{v}|)$ filter | $\leq |\mathbf{v}|$ |
| Pauli noise | $O(n/W)$ | **zero** | unchanged |